# Separação de equipas + seguimento — jogo inteiro no Colab

**Antes de começar**
1. No teu Google Drive cria a pasta `futebol` e carrega para lá:
   - o vídeo `AFL-3Dv-S1-SnapYT.App.mp4`
   - o ficheiro `pacote.zip` (código, referências do estádio e modelo)
2. Aqui no Colab: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
3. Corre as células por ordem.

Se a sessão cair, volta a correr tudo: os trechos já feitos ficam no Drive e são saltados.

## 1. Verificar a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ligar o Google Drive e definir caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA   = '/content/drive/MyDrive/futebol'          # muda se usaste outra pasta
VIDEO   = f'{PASTA}/AFL-3Dv-S1-SnapYT.App.mp4'
PACOTE  = f'{PASTA}/pacote.zip'
SAIDA   = f'{PASTA}/resultado'                       # CSV e vídeos ficam aqui (no Drive)

FPS     = 5        # frames por segundo analisados (5 = seguimento fiável)
TRECHO  = 300      # segundos por trecho (unidade de retoma)
VIDEOS  = True     # grava um vídeo por trecho com caixas, IDs e equipa (~70 MB por trecho)
T0, T1  = 0, None  # início/fim em segundos (None = até ao fim)

## 3. Instalar dependências e preparar o código

In [ ]:
!pip -q install "ultralytics==8.4.192"
!rm -rf /content/jogo && mkdir -p /content/jogo && unzip -q -o "$PACOTE" -d /content/jogo && ls /content/jogo
# cópia local do vídeo (ler do Drive diretamente é mais lento)
!cp -n "$VIDEO" /content/video.mp4 && ls -lh /content/video.mp4

## 4. Processar
Mostra o progresso por trecho. Estimativa (não medida no Colab): ~2 h para o jogo inteiro a 5 fps;
a deteção na GPU é rápida, o resto (cor, registo nas referências, seguimento) corre no CPU.

In [ ]:
extra = '--video' if VIDEOS else ''
t1 = f'--t1 {T1}' if T1 else ''
!cd /content/jogo && python process.py /content/video.mp4 "$SAIDA" --fps $FPS --chunk $TRECHO --t0 $T0 $t1 $extra 2>&1 | grep -v -i warn

## 5. Ver um trecho (vídeo) e o resumo

In [ ]:
import glob, subprocess, base64, pandas as pd
from IPython.display import HTML, display
vids = sorted(glob.glob(f'{SAIDA}/chunks/*.mp4'))
print(len(vids), 'vídeos de trecho')
if vids:
    v = vids[0]                                        # muda o índice para ver outro trecho
    subprocess.run(['ffmpeg','-v','error','-y','-i',v,'-c:v','libx264','-crf','28','-preset','veryfast','-pix_fmt','yuv420p','/content/ver.mp4'], check=True)
    display(HTML(f'<video width=900 controls src="data:video/mp4;base64,{base64.b64encode(open("/content/ver.mp4","rb").read()).decode()}"></video>'))

d = pd.read_csv(f'{SAIDA}/jogadores.csv')
g = d.groupby('track').agg(equipa=('equipa','first'), ini=('t','min'), fim=('t','max'))
print('trajetórias por equipa:', g.equipa.value_counts().to_dict())
print('linhas (pessoa x frame):', len(d))
d.head()

## Saídas
- `resultado/jogadores.csv`: uma linha por pessoa seguida por frame:
  `t` (s), `track` (ID da trajetória, por trecho), `equipa` (rótulo final da trajetória: vermelho / rosa / arbitro / gr),
  `equipa_frame` (rótulo só desse frame), `x_pes`, `y_pes` (posição dos pés **em píxeis da imagem**), caixa `x1..y2`.
- `resultado/chunks/`: CSV e vídeo de cada trecho.

Limite atual: posições em píxeis (sem mapa do campo), IDs recomeçam em cada trecho e partem quando o jogador sai do enquadramento.